# Milestone 4

In [2]:
import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer,AutoModelForMultipleChoice,TrainingArguments,Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset

e:\IIT M DS\DL-GENAI Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
CHOICES = ["A", "B", "C", "D", "E"]
LABEL2IDX  = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128

In [5]:
train = pd.read_csv("../dataset/train.csv")
train[CHOICES] = train[CHOICES].fillna("").astype(str)
train["prompt"] = train["prompt"].fillna("").astype(str)
train["answer"] = train["answer"].str.strip().str.upper()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [6]:
train["label"] = train["answer"].map(LABEL2IDX)

label_150 = train.iloc[150]["label"]
answer_150 = train.iloc[150]["answer"]

print(f"Row 150 answer letter: {answer_150}")
print(f"Encoded label: {int(label_150)}")

Row 150 answer letter: C
Encoded label: 2


In [7]:
row0 = train.iloc[0]
prompt_0 = str(row0["prompt"])
option_b0 = str(row0["B"])

formatted = prompt_0 + " [SEP] " + option_b0
char_len = len(formatted)

print(f"Prompt: {prompt_0[:60]}...")
print(f"Option B: {option_b0}")
print(f"Formatted: {formatted[:80]}...")
print(f"Character length: {char_len}")

Prompt: Pick the best possible answer: What is Martin Heidegger's vi...
Option B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Formatted: Pick the best possible answer: What is Martin Heidegger's view on the relationsh...
Character length: 407


In [8]:
def format_options(row):
    prompt = str(row["prompt"])
    return [prompt + " [SEP] " + str(row[c]) for c in CHOICES]

In [9]:
options_row0 = format_options(row0)

encoding_row0 = tokenizer(
    options_row0,
    padding = "max_length",
    truncation = True,
    max_length = MAX_LEN,
    return_tensors= "pt",
)

input_ids_row0 = encoding_row0["input_ids"].unsqueeze(0)
second_dim = input_ids_row0.shape[1]

print(f"input_ids shape: {input_ids_row0.shape}")
print(f"Second dimension (num choices): {second_dim}")

input_ids shape: torch.Size([1, 5, 128])
Second dimension (num choices): 5


In [10]:
batch16 = train.iloc[:16]

all_input_ids = []
for _, row in batch16.iterrows():
    opts = format_options(row)
    enc  = tokenizer(
        opts,
        padding = "max_length",
        truncation = True,
        max_length = MAX_LEN,
        return_tensors= "pt",
    )
    all_input_ids.append(enc["input_ids"].unsqueeze(0))

batch_input_ids = torch.cat(all_input_ids, dim=0)
total_positions = batch_input_ids.numel()

print(f"Batch input_ids shape : {batch_input_ids.shape}")
print(f"Total token positions: {total_positions}")

Batch input_ids shape : torch.Size([16, 5, 128])
Total token positions: 10240


In [ ]:
mc_model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
mc_model.eval()

options_row0 = format_options(row0)
enc_row0 = tokenizer(
    options_row0,
    padding = "max_length",
    truncation = True,
    max_length = MAX_LEN,
    return_tensors = "pt",
)
input_ids_mc = enc_row0["input_ids"].unsqueeze(0)
attention_mask_mc = enc_row0["attention_mask"].unsqueeze(0)

with torch.no_grad():
    outputs_q5 = mc_model(
        input_ids = input_ids_mc,
        attention_mask = attention_mask_mc,
    )

logits_shape = outputs_q5.logits.shape
num_logits = logits_shape[-1]

print(f"Logits tensor shape : {logits_shape}")
print(f"Number of logits per question: {num_logits}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1927.63it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Cons

Logits tensor shape : torch.Size([1, 5])
Number of logits per question: 5


In [12]:
label_row0 = torch.tensor([int(train.iloc[0]["label"])], dtype=torch.long)

with torch.no_grad():
    outputs_q6 = mc_model(
        input_ids = input_ids_mc,
        attention_mask = attention_mask_mc,
        labels = label_row0,
    )

loss_tensor = outputs_q6.loss
num_dims = loss_tensor.dim()

print(f"Loss tensor : {loss_tensor}")
print(f"Loss tensor shape : {loss_tensor.shape}")
print(f"Number of dimensions: {num_dims}")

Loss tensor : 1.6459709405899048
Loss tensor shape : torch.Size([])
Number of dimensions: 0


In [13]:
lora_base = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    r = 8,
    lora_alpha = 16,
    target_modules = ["query", "value"],
    lora_dropout = 0.1,
    bias = "none",
    task_type = TaskType.SEQ_CLS,
)

lora_model = get_peft_model(lora_base, lora_config)

trainable_params = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in lora_model.parameters())

print(f"All parameters: {all_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable %: {100 * trainable_params / all_params:.4f}%")
print(f"Trainable parameters: {trainable_params}")

lora_model.print_trainable_parameters()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3204.17it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Cons

All parameters: 109,778,690
Trainable parameters: 295,681
Trainable %: 0.2693%
Trainable parameters: 295681
trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


In [14]:
def tokenize_row(row):
    opts = [str(row["prompt"]) + " [SEP] " + str(row[c]) for c in CHOICES]
    enc  = tokenizer(
        opts,
        padding = "max_length",
        truncation = True,
        max_length = MAX_LEN,
    )
    return {
        "input_ids" : enc["input_ids"],
        "attention_mask" : enc["attention_mask"],
        "labels" : LABEL2IDX[row["answer"]],
    }

rows_100 = [tokenize_row(train.iloc[i]) for i in range(100)]
hf_dataset = Dataset.from_list(rows_100)

first_item = hf_dataset[0]
input_ids_shape = len(first_item["input_ids"]), len(first_item["input_ids"][0])
num_choices = input_ids_shape[0]

print(f"First item input_ids shape : {input_ids_shape}")
print(f"Dataset size: {len(hf_dataset)}")
print(f"Tokenized choices in input_ids: {num_choices}")

First item input_ids shape : (5, 128)
Dataset size: 100
Tokenized choices in input_ids: 5


In [15]:
MAX_LEN_TINY = 64

def tokenize_row_tiny(row):
    opts = [str(row["prompt"]) + " [SEP] " + str(row[c]) for c in CHOICES]
    enc  = tokenizer(
        opts,
        padding = "max_length",
        truncation = True,
        max_length = MAX_LEN_TINY,
    )
    return {
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "labels": LABEL2IDX[row["answer"]],
    }

rows_32 = [tokenize_row_tiny(train.iloc[i]) for i in range(32)]
tiny_ds = Dataset.from_list(rows_32)

tiny_ds = tiny_ds.with_format("torch")

lora_base_q9 = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
lora_model_q9 = get_peft_model(lora_base_q9, LoraConfig(
    r = 8,
    lora_alpha = 16,
    target_modules = ["query", "value"],
    lora_dropout = 0.1,
    bias = "none",
    task_type = TaskType.SEQ_CLS,
))

training_args = TrainingArguments(
    output_dir = "./lora_tiny_output",
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 1,
    max_steps = 4,
    logging_steps = 1,
    save_steps = 999,
    report_to = "none",
    fp16 = torch.cuda.is_available(),
)
trainer = Trainer(
    model = lora_model_q9,
    args = training_args,
    train_dataset = tiny_ds,
)

train_result = trainer.train()
global_step = train_result.global_step

print(f"Final global_step: {global_step}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4047.02it/s]
[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Cons

Step,Training Loss
1,1.624268
2,1.703369
3,1.671143
4,1.668945


Final global_step: 4


In [16]:
lora_model_q9.eval()

opts_row0 = [str(row0["prompt"]) + " [SEP] " + str(row0[c]) for c in CHOICES]
enc_q10 = tokenizer(
    opts_row0,
    padding = "max_length",
    truncation = True,
    max_length = MAX_LEN_TINY,
    return_tensors= "pt",
)

input_ids_q10 = enc_q10["input_ids"].unsqueeze(0)
attention_mask_q10 = enc_q10["attention_mask"].unsqueeze(0)

device = next(lora_model_q9.parameters()).device
input_ids_q10 = input_ids_q10.to(device)
attention_mask_q10 = attention_mask_q10.to(device)

with torch.no_grad():
    outputs_q10 = lora_model_q9(
        input_ids = input_ids_q10,
        attention_mask = attention_mask_q10,
    )

logits_q10 = outputs_q10.logits
probs_q10  = torch.softmax(logits_q10, dim=-1).squeeze(0)

print(f"Logits : {logits_q10}")
print(f"Probabilities per option:")
for i, c in enumerate(CHOICES):
    print(f"Option {c}: {probs_q10[i].item():.4f}")

prob_E = probs_q10[4].item()
print(f"Probability of Option E: {round(prob_E, 4)}")

Logits : tensor([[-0.0414,  0.0371, -0.0434, -0.1198, -0.1335]], device='cuda:0')
Probabilities per option:
Option A: 0.2034
Option B: 0.2200
Option C: 0.2030
Option D: 0.1881
Option E: 0.1855
Probability of Option E: 0.1855
